# The JAX/XLA Stack — No Black Boxes (hands-on)

A companion notebook for the lesson [*The JAX/XLA Stack — No Black Boxes*](https://lms-p-45c03.web.app/topics/ml-systems/jax-xla-stack/).

You'll **run and measure** the ideas from the lesson instead of just reading them: the
**compile path** (trace → StableHLO → XLA), **fusion** (the same chain the simulation
animates), the **arithmetic-intensity roofline**, and the **AOT trick** that catches an
out-of-memory error *before* you spend a slice.

**It runs fully on CPU** — every cell works on the default runtime, because XLA fuses and
compiles the *same* program for every backend. The closing benchmark *also* lights up on a
free TPU: `Runtime → Change runtime type → v5e-1 TPU`.

> Tested with JAX 0.4.x. Run the cells top to bottom.

## 1. What hardware did you get?
JAX runs the *same* code on CPU, GPU, or TPU — only the backend changes.

In [ ]:
# Colab ships JAX preinstalled on CPU and TPU runtimes.
# To pin a known-good version, uncomment:  !pip install -q "jax==0.4.30"
import time, math
import numpy as np
import jax, jax.numpy as jnp

print("JAX", jax.__version__)
devices = jax.devices()
PLATFORM = devices[0].platform            # 'cpu', 'gpu', or 'tpu'
HAS_TPU = PLATFORM == "tpu"
print("Devices :", devices)
print("Backend :", PLATFORM.upper(), "| TPU available:", HAS_TPU)
if not HAS_TPU:
    print("\nThis notebook runs fully on CPU. For the real-hardware benchmark,")
    print("select: Runtime -> Change runtime type -> v5e-1 TPU.")

## 2. The compile path: trace → StableHLO → XLA

You write NumPy-like Python; the chip runs machine code. The first bridge is the **trace**:
when you call a `jax.jit` function, JAX runs it once with *abstract* values to record a
static graph (a **jaxpr**) of the operations. Here's our one layer — a matmul followed by
the cheap elementwise chain (bias → GELU → scale) — and the graph JAX records for it.

In [ ]:
N = 512                                    # one 512x512 layer, bf16 (matches the simulation)
x = jax.random.normal(jax.random.PRNGKey(0), (N, N), dtype=jnp.bfloat16)
w = jax.random.normal(jax.random.PRNGKey(1), (N, N), dtype=jnp.bfloat16)
b = jnp.zeros((N,), dtype=jnp.bfloat16)
s = jnp.bfloat16(1.25)

def layer(x, w, b, s):
    y = x @ w               # MatMul  - the heavy op
    y = y + b               # + bias   |
    y = jax.nn.gelu(y)      # GELU     |  a chain of cheap elementwise ops
    y = y * s               # x scale  |
    return y

# Trace: JAX records a static graph (jaxpr) by running once with abstract values.
print(jax.make_jaxpr(layer)(x, w, b, s))

Next the jaxpr is **lowered to StableHLO** — a portable, hardware-neutral IR. This is the
*exact* program XLA receives, and it compiles for CPU, GPU, or TPU unchanged.

In [ ]:
stablehlo = jax.jit(layer).lower(x, w, b, s).as_text()
print(stablehlo[:850])
print("\n... (truncated) — same StableHLO, any backend.")

## 3. Fusion you can see

This is the optimization the simulation animates. **Fused**, XLA merges bias/GELU/scale into
the matmul's epilogue, so the intermediates never leave the chip. **Unfused**, each op is its
own kernel and every intermediate is written to and re-read from memory — the HBM round-trips.

We build both, confirm they compute *identical* math, then read **XLA's own cost model** to
see the byte traffic drop.

In [ ]:
fused = jax.jit(layer)

# Unfused: each op is its OWN jitted kernel -> intermediates round-trip through memory.
mm    = jax.jit(lambda x, w: x @ w)
addb  = jax.jit(lambda y, b: y + b)
gelu  = jax.jit(lambda y: jax.nn.gelu(y))
scale = jax.jit(lambda y, s: y * s)
def unfused(x, w, b, s):
    return scale(gelu(addb(mm(x, w), b)), s)

diff = float(jnp.max(jnp.abs(fused(x, w, b, s) - unfused(x, w, b, s))))
print("max |fused - unfused| =", diff, " -> identical math\n")

def cost(compiled):
    ca = compiled.cost_analysis()
    if isinstance(ca, (list, tuple)): ca = ca[0]
    return ca.get("flops", 0), ca.get("bytes accessed", 0)

ff, bf = cost(fused.lower(x, w, b, s).compile())
y0 = x @ w
parts = [mm.lower(x, w).compile(), addb.lower(y0, b).compile(),
         gelu.lower(y0).compile(), scale.lower(y0, s).compile()]
fu = sum(cost(p)[0] for p in parts)
bu = sum(cost(p)[1] for p in parts)

print(f"FUSED   : {ff/1e6:6.1f} MFLOP   {bf/1e6:6.2f} MB moved   intensity {ff/bf:5.1f} FLOP/byte")
print(f"UNFUSED : {fu/1e6:6.1f} MFLOP   {bu/1e6:6.2f} MB moved   intensity {fu/bu:5.1f} FLOP/byte")
print(f"\n-> Same FLOPs; fused moves ~{(1-bf/bu)*100:.0f}% fewer bytes -> higher arithmetic intensity.")
print("   (On CPU the gap is modest; on a TPU, where HBM bandwidth is the wall, it is dramatic.)")

## 4. The roofline — the simulation's exact numbers

XLA's cost model counts every byte on your backend. The lesson's simulation uses the idealized
**v5p** picture: one 512×512 bf16 tensor = **0.5 MB**, so you can just count HBM trips. Reproduce
it and watch the layer cross the ridge.

In [ ]:
TENSOR_MB = 0.5                 # a 512x512 bf16 tensor
FLOPS     = 2 * N**3            # matmul dominates ~ 268 MFLOP
RIDGE     = 165                 # v5p ridge: 459 TFLOP/s / 2.8 TB/s, in FLOP/byte

for name, trips in [("Unfused", 9), ("Fused", 3)]:
    mb = trips * TENSOR_MB
    ai = FLOPS / (mb * 1e6)
    bound = "compute-bound" if ai > RIDGE else "MEMORY-bound"
    print(f"{name:8}: {trips} HBM trips   {mb:.1f} MB   {ai:5.0f} FLOP/byte   -> {bound}")

print(f"\nv5p ridge = {RIDGE} FLOP/byte. Fusion collapses 9 trips -> 3, lifting the layer")
print("from below the ridge (memory-starved, MXU idle) to above it (MXU is the bottleneck).")

## 5. AOT — catch an OOM *before* you spend a slice

`jax.jit(f).lower(args).compile()` compiles **without running**. Lower from **abstract shapes**
(`jax.ShapeDtypeStruct`) and *nothing is allocated* — yet the compiled object already knows the
memory footprint. That's the level's "no black box" payoff: you discover a batch size will OOM a
v5p slice at **compile time**, before a single step burns real (and scarce) capacity.

In [ ]:
def hbm_needed_GB(batch, d_model=8192, dtype=jnp.bfloat16):
    shapes = (jax.ShapeDtypeStruct((batch, d_model), dtype),   # x
              jax.ShapeDtypeStruct((d_model, d_model), dtype),  # w
              jax.ShapeDtypeStruct((d_model,), dtype),          # b
              jax.ShapeDtypeStruct((), dtype))                  # s
    m = jax.jit(layer).lower(*shapes).compile().memory_analysis()
    return (m.temp_size_in_bytes + m.output_size_in_bytes + m.argument_size_in_bytes) / 1e9

V5P_HBM_GB = 95
print(f"v5p HBM budget ~ {V5P_HBM_GB} GB\n")
for batch in (4096, 524288, 1048576):
    gb = hbm_needed_GB(batch)
    print(f"batch={batch:>8}   needs {gb:7.1f} GB   -> {'OOM at compile time!' if gb > V5P_HBM_GB else 'fits'}")

print("\nNo array was ever allocated. The compiler told you the footprint before any step ran.")

## 6. A fair benchmark: fused vs unfused

Warm up first (the first call includes XLA **compilation**, which you must not time), then take
the median. Fusing removes kernel launches and HBM round-trips — a modest win on CPU, a big one
on a TPU whose MXU is fast enough that wasted memory trips dominate.

In [ ]:
def benchmark(f, device=None, iters=100):
    args = (x, w, b, s) if device is None else tuple(jax.device_put(a, device) for a in (x, w, b, s))
    f(*args).block_until_ready()                  # warmup = compile (NOT timed)
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); f(*args).block_until_ready(); ts.append(time.perf_counter() - t0)
    return sorted(ts)[len(ts) // 2] * 1e6          # median microseconds

mf, mu = benchmark(fused), benchmark(unfused)
print(f"FUSED   median  {mf:7.1f} us")
print(f"UNFUSED median  {mu:7.1f} us   ({mu/mf:.2f}x slower - extra kernel launches + HBM trips)")

if HAS_TPU:
    tpu = jax.devices("tpu")[0]
    tf_, tu_ = benchmark(fused, tpu), benchmark(unfused, tpu)
    print(f"\nTPU: fused {tf_:7.1f} us   unfused {tu_:7.1f} us   ({tu_/tf_:.2f}x)")
    print("Wider on TPU: the MXU is so fast that wasted HBM trips dominate the time.")
else:
    print("\n(Select a v5e-1 TPU runtime and re-run to see the gap widen on real silicon.)")

## Put it together

1. **Trace:** in §2, change `N` and re-print the jaxpr. Which line is the `dot_general` (the matmul), and which are the elementwise ops that fusion will absorb?
2. **Fusion:** in §3, why is `max |fused - unfused|` exactly `0.0` even though the byte traffic differs? What did fusion change — and what did it *not*?
3. **Roofline:** in §4, the fused layer reports ~179 FLOP/byte. Above the v5p ridge of 165 — so is it memory-bound or compute-bound, and which hardware unit is now the bottleneck?
4. **AOT:** in §5, find the batch size where `d_model=8192` first OOMs a v5p. Then raise `d_model` to 16384 — does the OOM happen sooner or later, and why?
5. **(TPU)** Grab a `v5e-1` runtime and run §6. How much wider is the fused-vs-unfused gap on the TPU than on CPU?

Back to the lesson → [The JAX/XLA Stack — No Black Boxes](https://lms-p-45c03.web.app/topics/ml-systems/jax-xla-stack/)